In [ ]:
# Weather benchmark: resource-aware + Spark 3.5

#Ноутбук сохраняет контракты `stations.json`, `cities.json` и `leaderboards.json`, а также схему сравнения 8 моделей, горизонтов и 21-дневного прогноза.

## Вычислительная схема

#- Jupyter/ML-контейнер ограничен **3 CPU и 6 GB RAM**. Локальные BLAS, scikit-learn, XGBoost и PyTorch ограничиваются одним потоком; модели не должны запускать `n_jobs=-1`.
#- Spark worker видит **3 cores и 8g Spark memory** при контейнерном лимите **3.5 CPU / 10 GB**. В Spark-клиенте выставляются `spark.cores.max=3`, `spark.executor.cores=3`, `spark.executor.memory=6g` и небольшое число shuffle partitions.
#- По умолчанию `WEATHER_EXECUTION_MODE=auto`: если доступен Spark Master, ноутбук проверяет Python и версии ML-пакетов на executor. Полные независимые city-задачи распределяются по Spark только при совместимом окружении; иначе бенчмарк безопасно выполняется локально, одним потоком.
#- Режим можно задать через `WEATHER_EXECUTION_MODE=local|auto|distributed`.

## Единый набор моделей (station и city совпадают)

#1. **Persistence** — ŷ(t+h) = y(t)
#2. **Seasonal Naive** — ŷ(t+h) = y(t+h−365)
#3. **SARIMAX** — ARMA(2,0,1) на аномалиях, рекурсивный прогноз
#4. **Ridge** — отдельная Ridge(α=10) на каждый горизонт
#5. **RandomForest** — 100 деревьев, глубина 12, max_features=0.5
#6. **XGBoost** — 300 деревьев, learning_rate=0.04, глубина 4
#7. **MLP** — одна нейросеть (feed-forward) с окном 30 дней и direct multi-output
#8. **LSTM** — одна рекуррентная сеть с lookback=30 и direct multi-output

#Master UI: `http://<server>:8085`, Worker UI: `http://<server>:8086`, History Server: `http://<server>:18080`.
#Jupyter по умолчанию ищет Master как `spark://spark-master:7077`.

In [ ]:
# ============================================================
# Setup: строгий ресурсный бюджет + необязательный Spark-клиент
# ============================================================
import os
import sys
import math
import json
import socket
import warnings
from pathlib import Path
from datetime import datetime

# Ограничить многопоточные библиотеки до импорта numpy/sklearn/torch.
for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
              "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[_name] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

try:
    from threadpoolctl import threadpool_limits
    threadpool_limits(limits=1)
except Exception:
    pass

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception as _e:
    HAS_XGB = False
    print(f"ℹ XGBoost недоступен; будет использоваться совместимый sklearn fallback: {_e}")

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_SARIMAX = True
except Exception as _e:
    HAS_SARIMAX = False
    print(f"ℹ SARIMAX недоступен: {_e}")

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except Exception:
        pass
except Exception as _e:
    HAS_TORCH = False
    print(f"ℹ PyTorch/LSTM недоступен: {_e}")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110


def _read_cgroup_cpu_limit():
    """Best-effort CPU quota reader for cgroup v1/v2."""
    try:
        p = Path("/sys/fs/cgroup/cpu.max")
        if p.exists():
            quota, period = p.read_text().strip().split()[:2]
            if quota != "max" and int(period) > 0:
                return max(0.1, int(quota) / int(period))
    except Exception:
        pass
    try:
        q = Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us")
        p = Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us")
        if q.exists() and p.exists():
            quota, period = int(q.read_text()), int(p.read_text())
            if quota > 0 and period > 0:
                return max(0.1, quota / period)
    except Exception:
        pass
    return float(os.cpu_count() or 1)


def _read_cgroup_mem_limit():
    """Return RAM limit in bytes where cgroups expose one."""
    candidates = [Path("/sys/fs/cgroup/memory.max"),
                  Path("/sys/fs/cgroup/memory/memory.limit_in_bytes")]
    for p in candidates:
        try:
            if not p.exists():
                continue
            raw = p.read_text().strip()
            if raw == "max":
                continue
            value = int(raw)
            if 0 < value < (1 << 60):
                return value
        except Exception:
            continue
    return None


CPU_LIMIT = _read_cgroup_cpu_limit()
MEM_LIMIT_BYTES = _read_cgroup_mem_limit()
LOCAL_THREADS = max(1, min(1, int(math.floor(CPU_LIMIT))))
WEATHER_EXECUTION_MODE = os.getenv("WEATHER_EXECUTION_MODE", "auto").strip().lower()
if WEATHER_EXECUTION_MODE not in {"auto", "local", "distributed"}:
    raise ValueError("WEATHER_EXECUTION_MODE должен быть auto, local или distributed")
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077").strip()
SPARK_WORKER_CORES = max(1, int(os.getenv("WEATHER_SPARK_CORES", "3")))
SPARK_EXECUTOR_MEMORY = os.getenv("WEATHER_SPARK_EXECUTOR_MEMORY", "6g")

try:
    from threadpoolctl import threadpool_limits
    threadpool_limits(limits=LOCAL_THREADS)
except Exception:
    pass


def _spark_executor_probe(_unused):
    """Runs on an executor: detects Python/package compatibility before ML tasks."""
    import sys as _sys
    import importlib
    import importlib.metadata
    versions = {}
    for import_name, dist_name in [
        ("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn"),
        ("statsmodels", "statsmodels"), ("torch", "torch"), ("xgboost", "xgboost"),
    ]:
        try:
            mod = importlib.import_module(import_name)
            version = getattr(mod, "__version__", None)
            if version is None:
                try:
                    version = importlib.metadata.version(dist_name)
                except Exception:
                    version = None
            versions[import_name] = version
        except Exception:
            versions[import_name] = None
    return {"python": [int(_sys.version_info.major), int(_sys.version_info.minor)],
            "versions": versions}


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "requirements.txt").exists() and (parent / "notebooks").exists():
            return parent
    for parent in [p, *p.parents]:
        if (parent / "data" / "weather" / "cities").exists():
            return parent
    raise RuntimeError("Корень проекта не найден. Открой notebook из корня проекта.")


PROJECT = find_project_root()
DATA_DIR = PROJECT / "data" / "weather" / "cities"
ARTIFACTS_DIR = PROJECT / "artifacts" / "weather" / "cities"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

spark = None
SPARK_AVAILABLE = False
SPARK_DISTRIBUTED = False
SPARK_WORKER_PROBE = None
SPARK_WORKER_COMPATIBLE = False
SPARK_DISABLE_REASON = "режим local выбран явно" if WEATHER_EXECUTION_MODE == "local" else ""

if WEATHER_EXECUTION_MODE != "local":
    try:
        from urllib.parse import urlparse
        parsed_master = urlparse(SPARK_MASTER_URL)
        if parsed_master.scheme == "spark":
            host, port = parsed_master.hostname, parsed_master.port or 7077
            with socket.create_connection((host, port), timeout=1.5):
                pass
        elif not SPARK_MASTER_URL.startswith("local["):
            raise ValueError(f"Неподдерживаемый Spark master URL: {SPARK_MASTER_URL}")

        from pyspark.sql import SparkSession
        builder = (SparkSession.builder
            .appName("weather-benchmark-21day-resource-aware")
            .master(SPARK_MASTER_URL)
            .config("spark.executor.cores", str(SPARK_WORKER_CORES))
            .config("spark.cores.max", str(SPARK_WORKER_CORES))
            .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
            .config("spark.driver.memory", os.getenv("WEATHER_SPARK_DRIVER_MEMORY", "1g"))
            .config("spark.driver.maxResultSize", "512m")
            .config("spark.default.parallelism", str(SPARK_WORKER_CORES))
            .config("spark.sql.shuffle.partitions", str(max(4, SPARK_WORKER_CORES * 2)))
            .config("spark.task.cpus", "1")
            .config("spark.python.worker.reuse", "true")
            .config("spark.executorEnv.OMP_NUM_THREADS", "1")
            .config("spark.executorEnv.MKL_NUM_THREADS", "1")
            .config("spark.executorEnv.OPENBLAS_NUM_THREADS", "1")
            .config("spark.executorEnv.NUMEXPR_NUM_THREADS", "1"))
        _event_dir = Path("/opt/spark/events")
        if _event_dir.is_dir() and os.access(_event_dir, os.W_OK):
            builder = (builder
                .config("spark.eventLog.enabled", "true")
                .config("spark.eventLog.dir", "file:/opt/spark/events"))
        spark = builder.getOrCreate()
        spark.sparkContext.setLogLevel("WARN")
        if not str(spark.version).startswith("3.5."):
            raise RuntimeError(f"Ожидался Spark 3.5.x, подключён Spark {spark.version}")
        SPARK_AVAILABLE = True
    except Exception as _e:
        SPARK_DISABLE_REASON = f"Spark не подключён: {type(_e).__name__}: {_e}"
        try:
            if spark is not None:
                spark.stop()
        except Exception:
            pass
        spark = None
        SPARK_AVAILABLE = False

print("PROJECT             :", PROJECT)
print("DATA_DIR            :", DATA_DIR)
print("ARTIFACTS_DIR       :", ARTIFACTS_DIR)
print(f"driver limits       : CPU≈{CPU_LIMIT:.2f}, RAM={MEM_LIMIT_BYTES / (1024**3):.2f} GiB" if MEM_LIMIT_BYTES else f"driver CPU limit    : ≈{CPU_LIMIT:.2f}; RAM limit unknown")
print(f"local model threads : {LOCAL_THREADS}")
print(f"model backends      : xgboost={HAS_XGB}, sarimax={HAS_SARIMAX}, torch={HAS_TORCH}")
print(f"Spark               : {'connected ' + str(spark.version) if SPARK_AVAILABLE else SPARK_DISABLE_REASON or 'disabled'}")
if not HAS_SARIMAX or not HAS_TORCH:
    print("⚠ Контракт дашборда требует ровно 8 моделей: нужны statsmodels и torch. "
          "Ноутбук остановится с явным сообщением в конфигурационной ячейке, а не запишет неполный leaderboard.")

In [ ]:
# ============================================================
# Конфигурация эксперимента, чтение данных и проверка Spark-кластера
# ============================================================
TRAIN_END  = pd.Timestamp("2021-12-31")
TEST_START = pd.Timestamp("2022-01-01")
TEST_END   = pd.Timestamp("2023-12-31")

# Горизонты
STATION_HORIZONS = list(range(1, 15))
CITY_HORIZONS    = list(range(1, 8))
STATION_H_MAX    = max(STATION_HORIZONS)

# City dashboard: 45 дней факта + 21 день прогноза.
CITY_FORECAST_DAYS = 21
CITY_N              = 66
CITY_PAST           = CITY_N - CITY_FORECAST_DAYS
CITY_BACK_START     = CITY_PAST
CITY_ORIGIN         = CITY_BACK_START - 1
CITY_H_MAX          = CITY_N - CITY_ORIGIN - 1
assert CITY_H_MAX == CITY_FORECAST_DAYS
assert CITY_PAST == CITY_BACK_START == CITY_ORIGIN + 1
assert CITY_H_MAX >= max(CITY_HORIZONS)

VARS = ["temp", "humidity", "precipitation", "wind", "pressure"]
CITY_VAR_MAP = {"temp": "temp", "hum": "humidity", "rain": "precipitation",
                "wind": "wind", "pres": "pressure"}
CITY_VAR_UNITS = {"temp": "°C", "hum": "%", "rain": "мм", "wind": "м/с", "pres": "гПа"}

# Единый набор из 8 моделей для station и city.
BASE_MODELS = ["Persistence", "Seasonal Naive", "Ridge", "RandomForest", "XGBoost"]
TEMPORAL_MODELS = []
if HAS_SARIMAX:
    TEMPORAL_MODELS.append("SARIMAX")
if HAS_TORCH:
    TEMPORAL_MODELS.append("MLP")
    TEMPORAL_MODELS.append("LSTM")
STATION_MODELS = BASE_MODELS + TEMPORAL_MODELS
CITY_MODELS = BASE_MODELS + TEMPORAL_MODELS
STATION_BASELINES = ("Persistence", "Seasonal Naive")
CITY_BASELINES = ("Persistence", "Seasonal Naive")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Не найдена директория данных: {DATA_DIR}")
if not (DATA_DIR / "cities_daily.csv").exists():
    raise FileNotFoundError(f"Не найден файл: {DATA_DIR / 'cities_daily.csv'}")
if not (DATA_DIR / "cities_daily_meta.json").exists():
    raise FileNotFoundError(f"Не найден файл: {DATA_DIR / 'cities_daily_meta.json'}")

cities_daily = pd.read_csv(DATA_DIR / "cities_daily.csv", parse_dates=["date"])
cities_daily = cities_daily.sort_values(["city", "date"]).reset_index(drop=True)
meta = json.loads((DATA_DIR / "cities_daily_meta.json").read_text(encoding="utf-8"))

required_cols = {"date", "city", *VARS}
missing_cols = sorted(required_cols - set(cities_daily.columns))
if missing_cols:
    raise ValueError(f"В cities_daily.csv не хватает колонок: {missing_cols}")
if cities_daily.empty:
    raise ValueError("cities_daily.csv пуст")

CITY_NAMES = sorted(cities_daily["city"].dropna().unique().tolist())
CITY_INFO  = {c["city"]: c for c in meta.get("cities", []) if "city" in c}
if not CITY_NAMES:
    raise ValueError("В cities_daily.csv нет городов")

for _c in CITY_NAMES:
    _d = cities_daily.loc[cities_daily["city"] == _c, "date"]
    assert _d.is_unique and (_d.diff().dropna() == pd.Timedelta(days=1)).all(), \
        f"{_c}: в ряду есть пропуски/дубли дат"
_ref = cities_daily.loc[cities_daily["city"] == CITY_NAMES[0], "date"].reset_index(drop=True)
for _c in CITY_NAMES[1:]:
    _d = cities_daily.loc[cities_daily["city"] == _c, "date"].reset_index(drop=True)
    assert _d.equals(_ref), f"{_c}: сетка дат не совпадает с {CITY_NAMES[0]}"

if STATION_MODELS != CITY_MODELS:
    raise RuntimeError("Station и City benchmark должны использовать один набор моделей")
if len(STATION_MODELS) != 8:
    raise RuntimeError(
        "Для соблюдения JSON-контракта нужны ровно 8 моделей, включая SARIMAX, MLP и LSTM. "
        f"Сейчас: {STATION_MODELS}. Проверь зависимости statsmodels и torch в ML-образе."
    )


def _driver_package_versions():
    return {
        "numpy": getattr(np, "__version__", None),
        "pandas": getattr(pd, "__version__", None),
        "sklearn": __import__("sklearn").__version__,
        "statsmodels": getattr(__import__("statsmodels"), "__version__", None) if HAS_SARIMAX else None,
        "torch": getattr(torch, "__version__", None) if HAS_TORCH else None,
        "xgboost": getattr(xgb, "__version__", None) if HAS_XGB else None,
    }


def _check_spark_executor_compatibility():
    global SPARK_WORKER_PROBE, SPARK_WORKER_COMPATIBLE, SPARK_DISTRIBUTED, SPARK_DISABLE_REASON
    if not SPARK_AVAILABLE:
        return
    try:
        SPARK_WORKER_PROBE = spark.sparkContext.parallelize([0], 1).map(_spark_executor_probe).collect()[0]
        py_match = tuple(SPARK_WORKER_PROBE["python"]) == tuple(sys.version_info[:2])
        driver_versions = _driver_package_versions()
        worker_versions = SPARK_WORKER_PROBE["versions"]
        package_match = all(driver_versions.get(k) == worker_versions.get(k) for k in driver_versions)
        SPARK_WORKER_COMPATIBLE = bool(py_match and package_match)
        if not py_match:
            SPARK_DISABLE_REASON = (
                f"Python driver={sys.version_info.major}.{sys.version_info.minor}, "
                f"executor={SPARK_WORKER_PROBE['python'][0]}.{SPARK_WORKER_PROBE['python'][1]}"
            )
        elif not package_match:
            diff = {k: (driver_versions.get(k), worker_versions.get(k))
                    for k in driver_versions if driver_versions.get(k) != worker_versions.get(k)}
            SPARK_DISABLE_REASON = f"пакеты driver/executor отличаются: {diff}"
        else:
            SPARK_DISABLE_REASON = ""
        SPARK_DISTRIBUTED = SPARK_WORKER_COMPATIBLE and WEATHER_EXECUTION_MODE in {"auto", "distributed"}
        if WEATHER_EXECUTION_MODE == "distributed" and not SPARK_WORKER_COMPATIBLE:
            print("⚠ Запрошен distributed-режим, но executor несовместим; безопасный локальный fallback:", SPARK_DISABLE_REASON)
    except Exception as _e:
        SPARK_WORKER_COMPATIBLE = False
        SPARK_DISTRIBUTED = False
        SPARK_DISABLE_REASON = f"executor probe завершился ошибкой: {type(_e).__name__}: {_e}"
        try:
            spark.stop()
        except Exception:
            pass
        globals()["spark"] = None
        globals()["SPARK_AVAILABLE"] = False

_check_spark_executor_compatibility()

if SPARK_AVAILABLE:
    try:
        from pyspark.sql import functions as F
        spark_input = cities_daily[["city", "date", *VARS]].copy()
        spark_df = spark.createDataFrame(spark_input)
        _spark_stats = (spark_df.groupBy("city")
            .agg(F.count(F.lit(1)).alias("rows"),
                 F.countDistinct("date").alias("unique_dates"),
                 F.min("date").alias("min_date"), F.max("date").alias("max_date"))
            .orderBy("city").collect())
        _spark_rows = {r["city"]: r.asDict() for r in _spark_stats}
        for _c in CITY_NAMES:
            _row = _spark_rows[_c]
            if int(_row["rows"]) != int(_row["unique_dates"]):
                raise ValueError(f"Spark QA: дубли дат для {_c}: {_row}")
        print("Spark SQL QA       : OK —", len(_spark_rows), "городов,", sum(r["rows"] for r in _spark_rows.values()), "строк")
    except Exception as _e:
        print(f"⚠ Spark SQL QA пропущен; используем проверки pandas: {type(_e).__name__}: {_e}")
        SPARK_DISTRIBUTED = False
        SPARK_DISABLE_REASON = f"Spark SQL QA не прошёл: {_e}"

print(f"Загружено          : {cities_daily.shape}")
print(f"Города             : {CITY_NAMES}")
print(f"Период             : {cities_daily['date'].min().date()} → {cities_daily['date'].max().date()}")
print(f"Модели             : {STATION_MODELS}")
print(f"Spark SQL          : {'on' if SPARK_AVAILABLE else 'off'}")
print(f"Spark distributed  : {'on' if SPARK_DISTRIBUTED else 'off'}" + (f" ({SPARK_DISABLE_REASON})" if SPARK_DISABLE_REASON else ""))

In [ ]:
# ============================================================
#  Метрики (единый формат для обоих дашбордов)
# ============================================================

def safe_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """MAPE, пропускающий |y_true| < 1e-6.
    ВНИМАНИЕ: для осадков/ветра считается только по «ненулевым» дням и
    сильно завышен — ориентироваться на MAE / WAPE."""
    mask = np.abs(y_true) >= 1e-6
    if mask.sum() == 0:
        return 0.0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def wape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """WAPE = Σ|e| / Σ|y|, % — устойчивая замена MAPE для рядов с нулями."""
    denom = float(np.sum(np.abs(y_true)))
    return float(np.sum(np.abs(y_pred - y_true)) / denom * 100) if denom > 1e-9 else 0.0


def compute_metrics(y_true, y_pred, baseline_mae: float | None = None) -> dict:
    """Единый набор метрик, который ждёт фронт."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    n = min(len(y_true), len(y_pred))
    if n == 0:
        return dict(mae=0.0, rmse=0.0, r2=0.0, mape=0.0, wape=0.0,
                    bias=0.0, maxError=0.0, medae=0.0, skill=0.0)

    yt, yp = y_true[:n], y_pred[:n]
    err  = yp - yt
    abse = np.abs(err)

    mae   = float(np.mean(abse))
    rmse  = float(np.sqrt(np.mean(err ** 2)))
    ssr   = float(np.sum(err ** 2))
    sst   = float(np.sum((yt - yt.mean()) ** 2))
    r2    = float(1 - ssr / sst) if sst > 0 else 0.0
    mape  = safe_mape(yt, yp)
    wp    = wape(yt, yp)
    bias  = float(np.mean(err))
    mx    = float(np.max(abse)) if n else 0.0
    medae = float(np.median(abse))
    skill = float(1 - mae / baseline_mae) if baseline_mae and baseline_mae > 0 else 0.0

    return dict(mae=mae, rmse=rmse, r2=r2, mape=mape, wape=wp,
                bias=bias, maxError=mx, medae=medae, skill=skill)


def holidays_for(dates: pd.Series) -> list[dict]:
    """Мини-календарь праздников для reference-lines на графике станций."""
    out = []
    for d in dates:
        md = (d.month, d.day)
        if md == (1, 1):
            out.append({"date": d.strftime("%Y-%m-%d"), "icon": "🎄", "label": "Новый год"})
        elif md == (7, 1):
            out.append({"date": d.strftime("%Y-%m-%d"), "icon": "☀️", "label": "Середина года"})
    return out

In [ ]:
# ============================================================
#  Общий каркас: климатология → аномалии → признаки origin'а
# ============================================================
ANOM_LAGS = [0, 1, 2, 3, 5, 7, 10, 14, 21, 30]
ANOM_ROLL = [3, 7, 14, 30]
ANOM_STD  = [7, 30]
MIN_HIST  = 30


def get_series(city_name: str, var_src: str):
    sub = (cities_daily.loc[cities_daily["city"] == city_name, ["date", var_src]]
           .reset_index(drop=True))
    y = (pd.Series(sub[var_src].astype(float).values)
         .interpolate(limit_direction="both").values)
    return pd.DatetimeIndex(sub["date"]), y


def climatology(dates_train, y_train, smooth: int = 21) -> np.ndarray:
    """Климатология по doy (366 значений), сглаженная по кругу. Только train."""
    doy = pd.DatetimeIndex(dates_train).dayofyear.values
    clim = (pd.Series(y_train).groupby(doy).mean()
            .reindex(range(1, 367)).interpolate(limit_direction="both").values)
    pad = smooth
    ext = np.concatenate([clim[-pad:], clim, clim[:pad]])
    sm = pd.Series(ext).rolling(smooth, center=True, min_periods=1).mean().values
    return sm[pad:-pad]


def clim_at(clim: np.ndarray, dates) -> np.ndarray:
    return clim[pd.DatetimeIndex(dates).dayofyear.values - 1]


def origin_features(a: np.ndarray, dates: pd.DatetimeIndex,
                    clim_o: np.ndarray) -> pd.DataFrame:
    """Богатые признаки состояния погоды на origin t; только история ≤ t."""
    anom = pd.Series(a, dtype=float)
    raw = pd.Series(np.asarray(a, dtype=float) + np.asarray(clim_o, dtype=float))
    cols = {}

    for lag in sorted(set(ANOM_LAGS + [4, 6, 8, 9, 12, 18])):
        cols[f"a_lag_{lag}"] = anom.shift(lag).values
    for lag in [1, 2, 3, 5, 7, 10, 14]:
        cols[f"a_diff_{lag}"] = (anom - anom.shift(lag)).values
    for w in sorted(set(ANOM_ROLL + [5, 10, 21])):
        r = anom.rolling(w, min_periods=w)
        cols[f"a_mean_{w}"] = r.mean().values
        cols[f"a_min_{w}"] = r.min().values
        cols[f"a_max_{w}"] = r.max().values
    for w in sorted(set(ANOM_STD + [3, 14])):
        cols[f"a_std_{w}"] = anom.rolling(w, min_periods=w).std().values

    for lag in [0, 1, 2, 3, 5, 7, 10, 14, 21, 30]:
        cols[f"raw_lag_{lag}"] = raw.shift(lag).values
    for lag in [1, 2, 3, 5, 7, 14]:
        cols[f"raw_diff_{lag}"] = (raw - raw.shift(lag)).values
    for w in [3, 5, 7, 14, 21, 30]:
        r = raw.rolling(w, min_periods=w)
        cols[f"raw_mean_{w}"] = r.mean().values
        cols[f"raw_std_{w}"] = r.std().values
    for w in [7, 14, 30]:
        cols[f"raw_min_{w}"] = raw.rolling(w, min_periods=w).min().values
        cols[f"raw_max_{w}"] = raw.rolling(w, min_periods=w).max().values

    doy = dates.dayofyear.values
    for harmonic in [1, 2, 3]:
        cols[f"doy_sin_{harmonic}"] = np.sin(2 * np.pi * harmonic * doy / 365.25)
        cols[f"doy_cos_{harmonic}"] = np.cos(2 * np.pi * harmonic * doy / 365.25)
    cols["month_sin"] = np.sin(2 * np.pi * dates.month.values / 12.0)
    cols["month_cos"] = np.cos(2 * np.pi * dates.month.values / 12.0)
    cols["clim_o"] = clim_o
    return pd.DataFrame(cols)


def stack_X(Xb: np.ndarray, dates: pd.DatetimeIndex, clim: np.ndarray,
            idx: np.ndarray, h: int) -> np.ndarray:
    """Признаки origin + (h, сезонность целевой даты, климатология цели)."""
    idx = np.asarray(idx)
    tgt_doy = (dates[idx] + pd.Timedelta(days=h)).dayofyear.values
    extra = np.column_stack([
        np.full(len(idx), float(h)),
        np.sin(2 * np.pi * tgt_doy / 365.25),
        np.cos(2 * np.pi * tgt_doy / 365.25),
        clim[tgt_doy - 1],
    ])
    return np.hstack([Xb[idx], extra])


def postprocess(var_src: str, pred: np.ndarray) -> np.ndarray:
    """Физичный диапазон: осадки/ветер ≥ 0, влажность 0..100."""
    if var_src in ("precipitation", "wind"):
        return np.maximum(pred, 0.0)
    if var_src == "humidity":
        return np.clip(pred, 0.0, 100.0)
    return pred


_ANOM_CACHE: dict = {}


def get_anom(city_name: str, var_src: str, train_end=None):
    cutoff = pd.Timestamp(TRAIN_END if train_end is None else train_end)
    key = (city_name, var_src, cutoff.strftime("%Y-%m-%d"))
    if key not in _ANOM_CACHE:
        dts, yv = get_series(city_name, var_src)
        n_tr = int((dts <= cutoff).sum())
        if n_tr < MIN_HIST + 2:
            raise ValueError(f"Недостаточно истории для {city_name}/{var_src} до {cutoff.date()}")
        cl = climatology(dts[:n_tr], yv[:n_tr])
        _ANOM_CACHE[key] = (dts, yv, cl, yv - clim_at(cl, dts))
    return _ANOM_CACHE[key]


def extra_features(city_name: str, var_src: str, cross_city: bool = True,
                   train_end=None) -> pd.DataFrame:
    cols = {}
    for v in VARS:
        if v == var_src:
            continue
        _, level = get_series(city_name, v)
        s = pd.Series(get_anom(city_name, v, train_end=train_end)[3])
        raw = pd.Series(level, dtype=float)
        cols[f"{v}_a0"] = s.values
        cols[f"{v}_d1"] = s.diff(1).values
        cols[f"{v}_d3"] = s.diff(3).values
        cols[f"{v}_d7"] = s.diff(7).values
        cols[f"{v}_m3"] = s.rolling(3, min_periods=3).mean().values
        cols[f"{v}_m7"] = s.rolling(7, min_periods=7).mean().values
        cols[f"{v}_sd7"] = s.rolling(7, min_periods=7).std().values
        cols[f"{v}_level0"] = raw.values
        cols[f"{v}_level_d1"] = raw.diff(1).values
        cols[f"{v}_level_m3"] = raw.rolling(3, min_periods=3).mean().values

    if cross_city:
        for c in CITY_NAMES:
            if c == city_name:
                continue
            for v in VARS:
                dts, raw, cl, a_cross = get_anom(c, v, train_end=train_end)
                s = pd.Series(a_cross)
                raw_s = pd.Series(raw, dtype=float)
                cols[f"x_{c}_{v}_a0"] = s.values
                cols[f"x_{c}_{v}_d1"] = s.diff(1).values
                cols[f"x_{c}_{v}_d3"] = s.diff(3).values
                cols[f"x_{c}_{v}_m3"] = s.rolling(3, min_periods=3).mean().values
                cols[f"x_{c}_{v}_level0"] = raw_s.values
                cols[f"x_{c}_{v}_level_d1"] = raw_s.diff(1).values
    return pd.DataFrame(cols)


def make_ctx(city_name: str, var_src: str, H: int, multivar: bool = True,
             train_end=None, origins_override=None) -> dict:
    dates, y = get_series(city_name, var_src)
    cutoff = pd.Timestamp(TRAIN_END if train_end is None else train_end)
    n_train = int((dates <= cutoff).sum())
    n_end   = int((dates <= TEST_END).sum())
    if n_train < MIN_HIST + H + 1:
        raise ValueError(f"Недостаточно истории до {cutoff.date()} для H={H}")

    if origins_override is None:
        origins = np.arange(n_train - H, n_end - 1)
    else:
        origins = np.asarray(origins_override, dtype=int).reshape(-1)
        if len(origins) == 0 or np.any(origins < 0) or np.any(origins >= len(dates)):
            raise ValueError("origins_override содержит недопустимый индекс")
        if np.any(origins >= n_train):
            raise ValueError("origin должен быть не позже train_end: утечка будущего")

    clim     = climatology(dates[:n_train], y[:n_train])
    clim_all = clim_at(clim, dates)
    a        = y - clim_all
    feats    = origin_features(a, dates, clim_all)
    if multivar:
        feats = pd.concat([
            feats,
            extra_features(city_name, var_src, train_end=cutoff)
        ], axis=1)

    seq_extra = np.column_stack([
        get_anom(city_name, v, train_end=cutoff)[3] /
        (np.std(get_anom(city_name, v, train_end=cutoff)[3][:n_train]) + 1e-9)
        for v in VARS if v != var_src
    ]) if multivar else np.empty((len(y), 0))

    clim_tgt = np.empty((len(origins), H))
    for h in range(1, H + 1):
        tgt_doy = (dates[origins] + pd.Timedelta(days=h)).dayofyear.values
        clim_tgt[:, h - 1] = clim[tgt_doy - 1]

    return dict(city=city_name, var_src=var_src, H=H,
                y=y, a=a, dates=dates, clim=clim,
                Xb=feats.values.astype(float), cols=list(feats.columns),
                n_train=n_train, n_end=n_end, seq_extra=seq_extra,
                origins=origins, clim_tgt=clim_tgt, train_end=cutoff)


def eval_matrix(ctx: dict, pred: np.ndarray, base_mae_by_h: dict | None = None,
                with_points: bool = False) -> dict:
    base_mae_by_h = base_mae_by_h or {}
    y, o, dates = ctx["y"], ctx["origins"], ctx["dates"]
    n_train, n_end = ctx["n_train"], ctx["n_end"]
    res = {}
    for h in range(1, ctx["H"] + 1):
        tgt = o + h
        ok = (tgt >= n_train) & (tgt <= n_end - 1)
        yt = y[tgt[ok]]
        yp = pred[ok, h - 1]
        cell = {
            "metrics":   compute_metrics(yt, yp, baseline_mae=base_mae_by_h.get(h)),
            "resid_std": float(np.std(yp - yt)) if len(yt) else 0.0,
            "n":         int(len(yt)),
        }
        if with_points:
            t_dates = dates[tgt[ok]].strftime("%Y-%m-%d")
            o_dates = dates[o[ok]].strftime("%Y-%m-%d")
            cell["points"] = [
                {"date": td, "originDate": od,
                 "actual": float(a_), "predicted": float(p_)}
                for td, od, a_, p_ in zip(t_dates, o_dates, yt, yp)
            ]
        res[h] = cell
    return res


print(f"ANOM_LAGS = {ANOM_LAGS}")
print(f"ROLL      = {ANOM_ROLL}, STD = {ANOM_STD}")

In [ ]:
# ============================================================
#  Модели (общие для станций и городов) → predict(origins) → [n, H]
#
#  Единый реестр из 8 моделей:
#    1) Persistence       — реализовано в model_matrix
#    2) Seasonal Naive    — реализовано в model_matrix
#    3) SARIMAX           — build_sarimax
#    4) Ridge             — build_ridge
#    5) RandomForest      — build_tree / build_city_tree_direct
#    6) XGBoost           — build_tree / build_xgb_resid
#    7) MLP               — build_mlp (feed-forward, direct multi-output)
#    8) LSTM              — build_lstm (recurrent, direct multi-output)
# ============================================================
from sklearn.ensemble import HistGradientBoostingRegressor
from numpy.lib.stride_tricks import sliding_window_view


def _fit_ridges(ctx: dict, alpha: float = 10.0):
    Xb, a, n_train, H = ctx["Xb"], ctx["a"], ctx["n_train"], ctx["H"]
    fin = np.isfinite(Xb).all(axis=1) & (np.arange(len(Xb)) < n_train)
    scaler = StandardScaler().fit(Xb[fin])
    Xz = scaler.transform(np.nan_to_num(Xb))
    ridges = {}
    for h in range(1, H + 1):
        idx = np.arange(MIN_HIST, n_train - h)
        ridges[h] = Ridge(alpha=alpha).fit(Xz[idx], a[idx + h])
    return ridges, Xz


def build_ridge(ctx):
    ridges, Xz = _fit_ridges(ctx)
    H = ctx["H"]

    def predict(origins):
        origins = np.asarray(origins)
        return np.column_stack([ridges[h].predict(Xz[origins]) for h in range(1, H + 1)])
    return predict


def _make_tree(kind: str):
    if kind == "RandomForest":
        return RandomForestRegressor(n_estimators=100, max_depth=12, max_features=0.5,
                                     min_samples_leaf=10, n_jobs=LOCAL_THREADS, random_state=42)
    if kind == "XGBoost":
        if HAS_XGB:
            return xgb.XGBRegressor(n_estimators=300, learning_rate=0.04, max_depth=4,
                                    subsample=0.8, colsample_bytree=0.8,
                                    min_child_weight=10, tree_method="hist",
                                    n_jobs=LOCAL_THREADS, random_state=42, verbosity=0)
        return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04,
                                             max_depth=4, min_samples_leaf=20,
                                             random_state=42)
    raise ValueError(kind)


def build_tree(ctx, kind: str):
    Xb, a, dates, clim = ctx["Xb"], ctx["a"], ctx["dates"], ctx["clim"]
    n_train, H = ctx["n_train"], ctx["H"]
    models = {}
    for h in range(1, H + 1):
        idx = np.arange(MIN_HIST, n_train - h)
        X, y = stack_X(Xb, dates, clim, idx, h), a[idx + h]
        ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
        models[h] = _make_tree(kind).fit(X[ok], y[ok])

    def predict(origins):
        origins = np.asarray(origins)
        return np.column_stack([
            models[h].predict(stack_X(Xb, dates, clim, origins, h))
            for h in range(1, H + 1)
        ])
    return predict


def _fit_gbm_stacked(X, y):
    if HAS_XGB:
        return xgb.XGBRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
            tree_method="hist", n_jobs=LOCAL_THREADS, random_state=42, verbosity=0,
        ).fit(X, y)
    return HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.05, max_depth=5,
        min_samples_leaf=40, random_state=42,
    ).fit(X, y)


def build_xgb_resid(ctx):
    Xb, a, dates, clim = ctx["Xb"], ctx["a"], ctx["dates"], ctx["clim"]
    n_train, H = ctx["n_train"], ctx["H"]
    ridges, Xz = _fit_ridges(ctx)

    h_train = sorted({h for h in (set(range(1, 8)) | {9, 11, 14, 17, 21, H}) if h <= H})
    Xs, ys = [], []
    for h in h_train:
        idx = np.arange(MIN_HIST, n_train - h)
        X = stack_X(Xb, dates, clim, idx, h)
        y = a[idx + h] - ridges[h].predict(Xz[idx])
        ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
        Xs.append(X[ok]); ys.append(y[ok])
    gbm = _fit_gbm_stacked(np.vstack(Xs), np.concatenate(ys))

    def predict(origins):
        origins = np.asarray(origins)
        out = np.empty((len(origins), H))
        for h in range(1, H + 1):
            out[:, h - 1] = (ridges[h].predict(Xz[origins])
                             + gbm.predict(stack_X(Xb, dates, clim, origins, h)))
        return out
    return predict


# ---------- MLP: feed-forward, direct multi-output ----------
MLP_LOOKBACK = 30
MLP_HIDDEN   = 64
MLP_EPOCHS   = max(1, int(os.getenv("WEATHER_MLP_EPOCHS", "20")))
MLP_BATCH    = 256

if HAS_TORCH:
    class _MLPNet(nn.Module):
        def __init__(self, lookback: int, n_in: int, horizon: int, hidden: int = MLP_HIDDEN):
            super().__init__()
            self.flatten = nn.Flatten()
            self.fc = nn.Sequential(
                nn.Linear(lookback * n_in, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, horizon),
            )

        def forward(self, x):
            return self.fc(self.flatten(x))


def build_mlp(ctx):
    a, dates, n_train, H = ctx["a"], ctx["dates"], ctx["n_train"], ctx["H"]
    LB = MLP_LOOKBACK
    sd_a = float(np.std(a[:n_train])) + 1e-9
    az = a / sd_a
    doy = dates.dayofyear.values
    chan = np.column_stack([az, ctx["seq_extra"],
                            np.sin(2 * np.pi * doy / 365.25),
                            np.cos(2 * np.pi * doy / 365.25)]).astype(np.float32)
    wins = sliding_window_view(chan, LB, axis=0)
    wins = np.ascontiguousarray(wins.transpose(0, 2, 1))

    tr_orig = np.arange(LB - 1, n_train - H)
    Y = np.stack([az[tr_orig + h] for h in range(1, H + 1)], axis=1).astype(np.float32)
    Xtr = torch.from_numpy(wins[tr_orig - (LB - 1)])
    Ytr = torch.from_numpy(Y)

    torch.manual_seed(42)
    net = _MLPNet(LB, chan.shape[1], H)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MLP_EPOCHS)
    loss_fn = nn.MSELoss()
    n = len(Xtr)
    for _ in range(MLP_EPOCHS):
        net.train()
        perm = torch.randperm(n)
        for b in range(0, n, MLP_BATCH):
            ib = perm[b:b + MLP_BATCH]
            opt.zero_grad()
            loss_fn(net(Xtr[ib]), Ytr[ib]).backward()
            opt.step()
        sched.step()
    net.eval()

    def predict(origins):
        with torch.no_grad():
            xb = torch.from_numpy(wins[np.asarray(origins) - (LB - 1)])
            return net(xb).numpy().astype(float) * sd_a
    return predict


# ---------- LSTM: direct multi-output ----------
LSTM_LOOKBACK = 30
LSTM_HIDDEN   = 48
LSTM_EPOCHS   = max(1, int(os.getenv("WEATHER_LSTM_EPOCHS", "20")))
LSTM_BATCH    = 256

if HAS_TORCH:
    class _LSTMNet(nn.Module):
        def __init__(self, horizon: int, n_in: int = 3, hidden: int = LSTM_HIDDEN):
            super().__init__()
            self.lstm = nn.LSTM(n_in, hidden, batch_first=True)
            self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                      nn.Linear(hidden, horizon))

        def forward(self, x):
            o, _ = self.lstm(x)
            return self.head(o[:, -1, :])


def build_lstm(ctx):
    a, dates, n_train, H = ctx["a"], ctx["dates"], ctx["n_train"], ctx["H"]
    LB = LSTM_LOOKBACK
    sd_a = float(np.std(a[:n_train])) + 1e-9
    az = a / sd_a
    doy = dates.dayofyear.values
    chan = np.column_stack([az, ctx["seq_extra"],
                            np.sin(2 * np.pi * doy / 365.25),
                            np.cos(2 * np.pi * doy / 365.25)]).astype(np.float32)
    wins = sliding_window_view(chan, LB, axis=0)
    wins = np.ascontiguousarray(wins.transpose(0, 2, 1))

    tr_orig = np.arange(LB - 1, n_train - H)
    Y = np.stack([az[tr_orig + h] for h in range(1, H + 1)], axis=1).astype(np.float32)
    Xtr = torch.from_numpy(wins[tr_orig - (LB - 1)])
    Ytr = torch.from_numpy(Y)

    torch.manual_seed(42)
    net = _LSTMNet(H, n_in=chan.shape[1])
    opt = torch.optim.AdamW(net.parameters(), lr=3e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=LSTM_EPOCHS)
    loss_fn = nn.MSELoss()
    n = len(Xtr)
    for _ in range(LSTM_EPOCHS):
        net.train()
        perm = torch.randperm(n)
        for b in range(0, n, LSTM_BATCH):
            ib = perm[b:b + LSTM_BATCH]
            opt.zero_grad()
            loss_fn(net(Xtr[ib]), Ytr[ib]).backward()
            opt.step()
        sched.step()
    net.eval()

    def predict(origins):
        with torch.no_grad():
            xb = torch.from_numpy(wins[np.asarray(origins) - (LB - 1)])
            return net(xb).numpy().astype(float) * sd_a
    return predict


# ---------- SARIMAX ----------
SARIMAX_ORDER  = (2, 0, 1)
SARIMAX_WINDOW = 90


def build_sarimax(ctx):
    a, n_train, H = ctx["a"], ctx["n_train"], ctx["H"]
    res = SARIMAX(a[:n_train][-3000:], order=SARIMAX_ORDER,
                  enforce_stationarity=True,
                  enforce_invertibility=True).fit(disp=False)
    params = np.asarray(res.params)
    decay = np.exp(-np.arange(1, H + 1) / 3.0)

    def predict(origins):
        out = np.empty((len(origins), H))
        for k, t in enumerate(origins):
            w = a[max(0, t - SARIMAX_WINDOW + 1): t + 1]
            try:
                m = SARIMAX(w, order=SARIMAX_ORDER,
                            enforce_stationarity=True, enforce_invertibility=True)
                fc = np.asarray(m.filter(params).forecast(steps=H), dtype=float)
                if not np.all(np.isfinite(fc)):
                    raise ValueError("NaN/Inf")
            except Exception:
                fc = a[t] * decay
            out[k] = fc
        return out
    return predict


def _make_city_tree(kind: str):
    if kind == "RandomForest":
        return RandomForestRegressor(
            n_estimators=30, max_depth=8, max_features="sqrt",
            min_samples_leaf=20, n_jobs=LOCAL_THREADS, random_state=42,
        )
    raise ValueError(kind)


def build_city_tree_direct(ctx: dict, kind: str):
    Xb, a = ctx["Xb"], ctx["a"]
    dates, clim = ctx["dates"], ctx["clim"]
    n_train, H = ctx["n_train"], ctx["H"]
    models = {}
    for h in range(1, H + 1):
        idx = np.arange(MIN_HIST, n_train - h)
        X = stack_X(Xb, dates, clim, idx, h)
        y = a[idx + h]
        ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
        if int(ok.sum()) < 50:
            raise ValueError(f"Недостаточно обучающих строк для {kind}, h={h}")
        models[h] = _make_city_tree(kind).fit(X[ok], y[ok])

    def predict(origins):
        origins = np.asarray(origins, dtype=int)
        return np.column_stack([
            models[h].predict(stack_X(Xb, dates, clim, origins, h))
            for h in range(1, H + 1)
        ])
    return predict


STATION_BUILDERS = {
    "Ridge":        build_ridge,
    "RandomForest": lambda c: build_tree(c, "RandomForest"),
    "XGBoost":      lambda c: build_tree(c, "XGBoost"),
}
if HAS_SARIMAX:
    STATION_BUILDERS["SARIMAX"] = build_sarimax
if HAS_TORCH:
    STATION_BUILDERS["MLP"]  = build_mlp
    STATION_BUILDERS["LSTM"] = build_lstm

CITY_BUILDERS = {
    "Ridge":        build_ridge,
    "RandomForest": lambda c: build_city_tree_direct(c, "RandomForest"),
    "XGBoost":      build_xgb_resid,
}
if HAS_SARIMAX:
    CITY_BUILDERS["SARIMAX"] = build_sarimax
if HAS_TORCH:
    CITY_BUILDERS["MLP"]  = build_mlp
    CITY_BUILDERS["LSTM"] = build_lstm

assert set(STATION_MODELS) == {"Persistence", "Seasonal Naive", *STATION_BUILDERS.keys()}
assert set(CITY_MODELS) == {"Persistence", "Seasonal Naive", *CITY_BUILDERS.keys()}
assert STATION_MODELS == CITY_MODELS, (STATION_MODELS, CITY_MODELS)


def model_matrix(model_name: str, ctx: dict, builders: dict) -> np.ndarray:
    o, y, H = ctx["origins"], ctx["y"], ctx["H"]
    if model_name == "Persistence":
        pred = np.repeat(y[o][:, None], H, axis=1)
    elif model_name == "Seasonal Naive":
        pred = np.column_stack([y[np.maximum(o + h - 365, 0)] for h in range(1, H + 1)])
    else:
        pred = ctx["clim_tgt"] + builders[model_name](ctx)(o)
    return postprocess(ctx["var_src"], pred)


print("Модели готовы:",
      "станции =", STATION_MODELS,
      "| города =", CITY_MODELS)

In [ ]:
# ============================================================
# Station benchmark: 4 станции × 8 моделей × 14 горизонтов
# ============================================================
import gc

print("=" * 64)
print(f"STATION BENCHMARK — {len(CITY_NAMES)} станций × {len(STATION_MODELS)} моделей × {len(STATION_HORIZONS)} горизонтов")
print("execution:", "Spark distributed" if SPARK_DISTRIBUTED else "local / one thread")
print("=" * 64)


def _cells_from_eval(ev: dict, base_mae: dict | None) -> dict:
    return {
        h: {
            "points": c["points"],
            "metrics": c["metrics"],
            "deltaVsPersistence": (c["metrics"]["mae"] - base_mae[h]) if base_mae else 0.0,
        }
        for h, c in ev.items()
    }


def _run_station_city(city_name: str):
    ctx = make_ctx(city_name, "temp", STATION_H_MAX)
    city_eval: dict[str, dict] = {}
    log_lines = [f"\n[{city_name}] n={len(ctx['y']):,} train≤{TRAIN_END.date()} test={TEST_START.date()}→{TEST_END.date()}"]

    pred_p = model_matrix("Persistence", ctx, {})
    ev_p = eval_matrix(ctx, pred_p, with_points=True)
    base_mae = {h: ev_p[h]["metrics"]["mae"] for h in ev_p}
    city_eval["Persistence"] = _cells_from_eval(ev_p, None)
    log_lines.append(f"  {'Persistence':16s} ok  MAE(h=7) = {ev_p[7]['metrics']['mae']:.3f}")

    for m in STATION_MODELS:
        if m == "Persistence":
            continue
        try:
            pred = model_matrix(m, ctx, STATION_BUILDERS)
            if not np.all(np.isfinite(pred)):
                raise ValueError("NaN/Inf в прогнозе")
            ev = eval_matrix(ctx, pred, base_mae, with_points=True)
            city_eval[m] = _cells_from_eval(ev, base_mae)
            log_lines.append(f"  {m:16s} ok  MAE(h=7) = {ev[7]['metrics']['mae']:.3f}  skill = {ev[7]['metrics']['skill']:+.3f}")
        except Exception as e:
            log_lines.append(f"  {m:16s} FAIL — {type(e).__name__}: {e}")

    missing_models = [m for m in STATION_MODELS if m not in city_eval]
    if missing_models:
        raise RuntimeError(f"{city_name}: не обучены модели {missing_models}; артефакты не обновлены")
    return city_name, city_eval, log_lines


station_results: dict[str, dict] = {}
if SPARK_DISTRIBUTED:
    try:
        _n_partitions = max(1, min(len(CITY_NAMES), SPARK_WORKER_CORES))
        _station_rows = (spark.sparkContext.parallelize(CITY_NAMES, _n_partitions)
                         .map(_run_station_city).collect())
    except Exception as _e:
        try:
            spark.sparkContext.cancelAllJobs()
        except Exception:
            pass
        print(f"⚠ Spark station run failed; falling back to local execution: {type(_e).__name__}: {_e}")
        SPARK_DISTRIBUTED = False
        SPARK_DISABLE_REASON = f"station Spark task failed: {_e}"
        _station_rows = [_run_station_city(c) for c in CITY_NAMES]
else:
    _station_rows = [_run_station_city(c) for c in CITY_NAMES]

for _city_name, _result, _logs in sorted(_station_rows, key=lambda x: x[0]):
    station_results[_city_name] = _result
    for _line in _logs:
        print(_line)
    del _result, _logs
    gc.collect()

print("\nГотово: все модели обучены для каждой станции.")

In [ ]:
# ============================================================
#  Сериализация station benchmark → stations.json
# ============================================================
SERIES_TAIL      = 200
MAX_POINTS_PER_H = 100


def _make_series(sub: pd.DataFrame) -> list[dict]:
    tail = sub.tail(SERIES_TAIL).copy()
    hol = {h["date"]: h for h in holidays_for(tail["date"])}
    out = []
    for _, row in tail.iterrows():
        d = row["date"].strftime("%Y-%m-%d")
        item = {"date": d, "actual": float(row["y"])}
        if d in hol:
            item["holiday"] = hol[d]["label"]
        out.append(item)
    return out


def _serialize_station_evals(evals: dict) -> dict:
    out = {}
    for h, cell in evals.items():
        m = cell.get("metrics", {})
        mae = float(m.get("mae", 0.0))
        all_pts = cell.get("points", [])
        trimmed = all_pts[-MAX_POINTS_PER_H:] if len(all_pts) > MAX_POINTS_PER_H else all_pts

        pts = []
        for p in trimmed:
            err = p["predicted"] - p["actual"]
            pts.append({
                "date": p["date"],
                "originDate": p.get("originDate"),
                "actual": round(p["actual"], 3),
                "predicted": round(p["predicted"], 3),
                "error": round(err, 3),
                "abs_error": round(abs(err), 3),
                "within_mae": abs(err) <= mae,
                "is_outlier": abs(err) > mae * 2.5,
            })
        out[str(h)] = {
            "horizon": h,
            "model": None,
            "points": pts,
            "pointsTotal": len(all_pts),
            "metrics": {k: round(float(v), 5) for k, v in m.items()},
            "deltaVsPersistence": round(float(cell.get("deltaVsPersistence", 0.0)), 5),
        }
    return out


feature_importance = []
try:
    _ctx = make_ctx(CITY_NAMES[0], "temp", STATION_H_MAX)
    idx = np.arange(MIN_HIST, _ctx["n_train"] - 1)
    X_fi, y_fi = _ctx["Xb"][idx], _ctx["a"][idx + 1]
    ok = np.isfinite(X_fi).all(axis=1) & np.isfinite(y_fi)
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=20,
                               n_jobs=LOCAL_THREADS, random_state=42).fit(X_fi[ok], y_fi[ok])
    pairs = sorted(zip(_ctx["cols"], rf.feature_importances_), key=lambda kv: -kv[1])[:10]
    total = sum(v for _, v in pairs) or 1.0
    feature_importance = [{"name": k, "value": round(float(v / total), 4)} for k, v in pairs]
except Exception as e:
    print("feature_importance fallback:", e)
    feature_importance = [{"name": c, "value": 0.0} for c in ["a_lag_0", "a_lag_1"]]


station_data = []
for city_name in CITY_NAMES:
    info = CITY_INFO[city_name]
    dts, yv = get_series(city_name, "temp")
    sub = pd.DataFrame({"date": dts, "y": yv})

    serialized_evals = {}
    for m in STATION_MODELS:
        if m not in station_results[city_name]:
            continue
        serialized_evals[m] = _serialize_station_evals(station_results[city_name][m])
        for h_key in serialized_evals[m]:
            serialized_evals[m][h_key]["model"] = m

    series = _make_series(sub)
    first_test_in_series = next((i for i, it in enumerate(series)
                                 if pd.Timestamp(it["date"]) >= TEST_START), len(series))

    station_data.append({
        "station": {
            "id": info["wmo_id"],
            "label": info["city_ru"],
            "lat": info["lat"],
            "lon": info["lon"],
            "meanT": float(sub["y"].mean()),
            "seasonalAmp": float(sub["y"].max() - sub["y"].min()),
            "noise": float(sub["y"].std() - (sub["y"].max() - sub["y"].min()) / 6),
        },
        "series": series,
        "testStart": int(first_test_in_series),
        "testStartDate": TEST_START.strftime("%Y-%m-%d"),
        "evaluations": serialized_evals,
    })


_STATION_MODEL_META = {
    "Persistence":      {"isBaseline": True,  "model": "ŷ(t+h)=y(t)", "description": "Последнее наблюдение"},
    "Seasonal Naive":   {"isBaseline": True,  "model": "ŷ(t+h)=y(t+h−365)", "description": "Сезонный прогноз по дню прошлого года"},
    "SARIMAX":          {"isBaseline": False, "model": "ARMA(2,0,1) на аномалиях", "description": "Статистическая модель временного ряда"},
    "Ridge":            {"isBaseline": False, "model": "Ridge(α=10) на аномалиях", "description": "Линейная регрессия L2"},
    "RandomForest":     {"isBaseline": False, "model": "RF(n=100, d=12) на аномалиях", "description": "Ансамбль деревьев"},
    "XGBoost":          {"isBaseline": False, "model": "XGBoost на аномалиях", "description": "Регуляризованный градиентный бустинг"},
    "MLP":              {"isBaseline": False, "model": "MLP direct multi-output (lookback=30)", "description": "Полносвязная нейросеть для оконных последовательностей"},
    "LSTM":             {"isBaseline": False, "model": "LSTM direct multi-output (lookback=30)", "description": "Рекуррентная нейросеть для последовательностей"},
}

stations_artifact = {
    "section": "forecasting-weather",
    "source": "open-meteo archive · real data",
    "generatedAt": datetime.now().isoformat(),
    "stations": [
        {"id": CITY_INFO[c]["wmo_id"], "label": CITY_INFO[c]["city_ru"]}
        for c in CITY_NAMES
    ],
    "models": [
        {"name": m, **_STATION_MODEL_META[m]} for m in STATION_MODELS
    ],
    "horizons": STATION_HORIZONS,
    "stationData": station_data,
    "featureImportance": feature_importance,
    "seriesLength": SERIES_TAIL,
    "pointsPerHorizon": MAX_POINTS_PER_H,
    "modelNames": list(STATION_MODELS),
    "trainCount": int((cities_daily["date"] <= TRAIN_END).sum() // len(CITY_NAMES)),
    "testDays": int(((cities_daily["date"] >= TEST_START) &
                     (cities_daily["date"] <= TEST_END)).sum() // len(CITY_NAMES)),
}

out_path = ARTIFACTS_DIR / "stations.json"
out_path.write_text(json.dumps(stations_artifact, ensure_ascii=False, separators=(",", ":")))
print(f"saved → {out_path}  ({out_path.stat().st_size/1024:.1f} KB)")

In [ ]:
# ============================================================
#  Визуализации station benchmark
# ============================================================
FIG_DIR = ARTIFACTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "Persistence":      "#94a3b8",
    "Seasonal Naive":   "#64748b",
    "Ridge":            "#818cf8",
    "RandomForest":     "#34d399",
    "XGBoost":          "#22d3ee",
    "SARIMAX":          "#a78bfa",
    "MLP":              "#fbbf24",
    "LSTM":             "#f472b6",
}

# ---------- 1. Skill(h) по горизонтам для каждой станции ----------
fig, axes = plt.subplots(1, len(CITY_NAMES), figsize=(5 * len(CITY_NAMES), 4.2),
                         sharey=True)
if len(CITY_NAMES) == 1:
    axes = [axes]

for ax, city_name in zip(axes, CITY_NAMES):
    per_model = station_results[city_name]
    for m in STATION_MODELS:
        if m not in per_model:
            continue
        model_evals = per_model[m]
        skills = []
        for h in STATION_HORIZONS:
            cell = model_evals.get(h) or model_evals.get(str(h))
            if cell is None or "metrics" not in cell:
                skills.append(np.nan)
            else:
                skills.append(cell["metrics"]["skill"])
        ax.plot(STATION_HORIZONS, skills, marker="o", ms=3.5, lw=1.6,
                color=PALETTE.get(m, "gray"),
                linestyle="--" if m in ("Persistence", "Seasonal Naive") else "-",
                label=m)
    ax.axhline(0, color="#f43f5e", lw=1, linestyle=":")
    ax.set_title(CITY_INFO[city_name]["city_ru"], fontsize=11)
    ax.set_xlabel("горизонт h, дней")
    ax.set_xticks([1, 3, 7, 10, 14])
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Skill = 1 − MAE_модели / MAE_persistence")
axes[-1].legend(loc="lower left", fontsize=8)
fig.suptitle("Skill(h) · 4 города · temp", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "stations_skill_by_horizon.png", dpi=110, bbox_inches="tight")
plt.show()


# ---------- 2. Средний MAE по моделям (bar) ----------
rows = []
for m in STATION_MODELS:
    cells = []
    for city_name in CITY_NAMES:
        per_model = station_results[city_name]
        if m not in per_model:
            continue
        model_evals = per_model[m]
        for h in STATION_HORIZONS:
            cell = model_evals.get(h) or model_evals.get(str(h))
            if cell and cell.get("points") and cell.get("metrics"):
                cells.append(cell["metrics"])
    if cells:
        rows.append({
            "model": m,
            "mae":   float(np.mean([c["mae"] for c in cells])),
            "rmse":  float(np.mean([c["rmse"] for c in cells])),
            "skill": float(np.mean([c["skill"] for c in cells])),
        })

df_st = pd.DataFrame(rows).sort_values("mae") if rows else pd.DataFrame()
if not df_st.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, label in zip(axes, ["mae", "rmse", "skill"],
                                  ["MAE, °C ↓", "RMSE, °C ↓", "Skill ↑"]):
        colors = [PALETTE.get(m, "gray") for m in df_st["model"]]
        ax.barh(df_st["model"], df_st[metric], color=colors)
        ax.invert_yaxis()
        ax.set_title(label, fontsize=11)
        ax.grid(alpha=0.3, axis="x")
        for i, v in enumerate(df_st[metric]):
            ax.text(v, i, f" {v:+.3f}" if metric == "skill" else f" {v:.3f}",
                    va="center", fontsize=8)
    fig.suptitle("Station benchmark · среднее по 4 станциям × 8 моделям × 14 горизонтам",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "stations_model_ranking.png", dpi=110, bbox_inches="tight")
    plt.show()


# ---------- 3. Predicted vs Actual: лучшая ML-модель (не baseline) на h=7 ----------
if not df_st.empty:
    _non_base = df_st[~df_st["model"].isin(STATION_BASELINES)]
    best_ml = (_non_base if not _non_base.empty else df_st).iloc[0]["model"]
    h_show = 7

    fig, axes = plt.subplots(1, len(CITY_NAMES), figsize=(5 * len(CITY_NAMES), 4.2),
                             sharex=False, sharey=False)
    if len(CITY_NAMES) == 1:
        axes = [axes]
    for ax, city_name in zip(axes, CITY_NAMES):
        model_evals = station_results[city_name][best_ml]
        cell = model_evals.get(h_show) or model_evals.get(str(h_show))
        pts = cell["points"] if cell else []
        if not pts:
            ax.set_title(f"{CITY_INFO[city_name]['city_ru']} · нет данных")
            continue
        yt = np.array([p["actual"] for p in pts])
        yp = np.array([p["predicted"] for p in pts])
        ax.scatter(yt, yp, s=6, alpha=0.35, color=PALETTE[best_ml])
        lim = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lim, lim, "--", color="#f43f5e", lw=1)
        ax.set_title(f"{CITY_INFO[city_name]['city_ru']} · MAE={np.mean(np.abs(yt-yp)):.2f}",
                     fontsize=10)
        ax.set_xlabel("Факт, °C")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Прогноз, °C")
    fig.suptitle(f"Predicted vs Actual · {best_ml} · h={h_show}", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "stations_pred_vs_actual.png", dpi=110, bbox_inches="tight")
    plt.show()


# ---------- 4. Распределение остатков ----------
if not df_st.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    for city_name in CITY_NAMES:
        model_evals = station_results[city_name][best_ml]
        cell = model_evals.get(h_show) or model_evals.get(str(h_show))
        pts = cell["points"] if cell else []
        if not pts:
            continue
        resid = np.array([p["predicted"] - p["actual"] for p in pts])
        ax.hist(resid, bins=40, alpha=0.45, label=CITY_INFO[city_name]["city_ru"])
    ax.axvline(0, color="#f43f5e", lw=1, linestyle="--")
    ax.set_xlabel("Остаток (прогноз − факт), °C")
    ax.set_ylabel("частота")
    ax.set_title(f"Распределение остатков · {best_ml} · h={h_show}")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "stations_residuals.png", dpi=110, bbox_inches="tight")
    plt.show()


# ---------- 5. Feature importance ----------
if feature_importance:
    fi = pd.DataFrame(feature_importance).sort_values("value")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(fi["name"], fi["value"], color="#818cf8")
    ax.set_xlabel("importance (нормировано)")
    ax.set_title("Feature importance · RandomForest · temp")
    ax.grid(alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "stations_feature_importance.png", dpi=110, bbox_inches="tight")
    plt.show()

print(f"\nФигуры сохранены в {FIG_DIR}")

In [ ]:
# ============================================================
# City benchmark: 4 города × 5 переменных × 8 моделей
# ============================================================
import time
import gc

print("=" * 64)
print("CITY BENCHMARK (origin-based, anomaly + direct)")
print("=" * 64)
print(f"backend: xgboost={HAS_XGB}, sarimax={HAS_SARIMAX}, torch={HAS_TORCH}")
print("execution:", "Spark distributed" if SPARK_DISTRIBUTED else "local / one thread")

latest = cities_daily["date"].max()
WINDOW_START = latest - pd.Timedelta(days=CITY_N - 1)
window = cities_daily[(cities_daily["date"] >= WINDOW_START) &
                      (cities_daily["date"] <= latest)].copy()
assert window["date"].nunique() == CITY_N, f"ожидалось {CITY_N} дней, есть {window['date'].nunique()}"
window_dates = [pd.Timestamp(d) for d in sorted(window["date"].unique())]
H_MAX = CITY_H_MAX
assert CITY_PAST == CITY_BACK_START == CITY_ORIGIN + 1
assert CITY_N - CITY_BACK_START == 21
EXPECTED_CITY_MODELS = list(STATION_MODELS)
assert CITY_MODELS == EXPECTED_CITY_MODELS, (CITY_MODELS, EXPECTED_CITY_MODELS)
assert len(STATION_MODELS) == len(CITY_MODELS) == 8, (
    f"Требуется 8 моделей в обоих benchmark: station={STATION_MODELS}, city={CITY_MODELS}"
)
assert CITY_MODELS == STATION_MODELS
assert len(CITY_MODELS) == len(set(CITY_MODELS))
print(f"Окно показа    : {window_dates[0].date()} → {window_dates[-1].date()} ({CITY_N} дней)")
print(f"Origin прогноза: {window_dates[CITY_ORIGIN].date()} → forecast {H_MAX} days")
print(f"Окно метрик    : {TEST_START.date()} → {TEST_END.date()}")


def _run_city_benchmark(city_name: str):
    vars_out = {}
    feature_importance_by_var = {}
    sigma_by_var = {}
    log_lines = [f"\n[{city_name}]"]

    for city_key, src_key in CITY_VAR_MAP.items():
        ctx = make_ctx(city_name, src_key, H_MAX)
        y, origins = ctx["y"], ctx["origins"]
        n_train, n_end = ctx["n_train"], ctx["n_end"]
        sigma_by_var[(city_name, city_key)] = float(np.std(y[n_train:n_end])) or 1.0

        idx_w = ctx["dates"].get_indexer(window_dates)
        assert (idx_w >= 0).all(), "в окне показа есть даты, которых нет в ряду"
        disp_idx = int(idx_w[CITY_ORIGIN])
        display_origin_date = pd.Timestamp(ctx["dates"][disp_idx])
        assert display_origin_date == window_dates[CITY_ORIGIN]

        ctx_display = make_ctx(
            city_name, src_key, H_MAX,
            train_end=display_origin_date,
            origins_override=np.array([disp_idx], dtype=int),
        )
        assert ctx_display["n_train"] == disp_idx + 1
        assert ctx_display["origins"].tolist() == [disp_idx]

        var_block = {
            "actual": [float(v) for v in y[idx_w]],
            "models": [],
            "leadMae": {},
            "metricsByHorizon": {},
        }

        pers_pred = model_matrix("Persistence", ctx, {})
        pers_ev = eval_matrix(ctx, pers_pred)
        pers_mae_by_h = {h: pers_ev[h]["metrics"]["mae"] for h in pers_ev}

        for mname in CITY_MODELS:
            if mname == "Persistence":
                pred, ev = pers_pred, pers_ev
            else:
                try:
                    pred = model_matrix(mname, ctx, CITY_BUILDERS)
                    if not np.all(np.isfinite(pred)):
                        raise ValueError("NaN/Inf в прогнозе")
                except Exception as e:
                    raise RuntimeError(
                        f"[{city_name}/{src_key}/{mname}] не удалось обучить модель; "
                        "останавливаем benchmark, не подменяя её Persistence."
                    ) from e
                ev = eval_matrix(ctx, pred, pers_mae_by_h)

            try:
                display_row = model_matrix(
                    mname, ctx_display,
                    CITY_BUILDERS if mname not in CITY_BASELINES else {}
                )[0]
                if not np.all(np.isfinite(display_row)):
                    raise ValueError("NaN/Inf в display-прогнозе")
            except Exception as e:
                raise RuntimeError(
                    f"[{city_name}/{src_key}/{mname}] не удалось построить display-прогноз; "
                    "не подменяем результат моделью Persistence."
                ) from e

            forecast = [None] * CITY_N
            sd_out = [None] * CITY_N
            for j in range(CITY_BACK_START, CITY_N):
                h = j - CITY_ORIGIN
                forecast[j] = float(display_row[h - 1])
                sd_out[j] = round(float(ev[h]["resid_std"]), 4)

            block = {
                "name": mname,
                "forecast": forecast,
                "sd": sd_out,
                "displayFitEnd": display_origin_date.strftime("%Y-%m-%d"),
            }
            block["metrics"] = {k: round(float(v), 5) for k, v in ev[1]["metrics"].items()}
            block["metricsWindow"] = {
                "start": TEST_START.strftime("%Y-%m-%d"),
                "end": TEST_END.strftime("%Y-%m-%d"),
                "n": int(ev[1]["n"]),
            }
            var_block["leadMae"][mname] = [
                round(float(ev[h]["metrics"]["mae"]), 4) for h in CITY_HORIZONS
            ]
            var_block["metricsByHorizon"][mname] = {
                str(h): {
                    **{k: round(float(v), 5) for k, v in ev[h]["metrics"].items()},
                    "n": ev[h]["n"],
                    "residStd": round(float(ev[h]["resid_std"]), 5),
                }
                for h in CITY_HORIZONS
            }
            var_block["models"].append(block)

        vars_out[city_key] = var_block

        try:
            idx = np.arange(MIN_HIST, n_train - 3)
            X_fi, y_fi = ctx["Xb"][idx], ctx["a"][idx + 3]
            ok = np.isfinite(X_fi).all(axis=1) & np.isfinite(y_fi)
            rf = RandomForestRegressor(n_estimators=60, max_depth=10, min_samples_leaf=20,
                                       n_jobs=LOCAL_THREADS, random_state=42).fit(X_fi[ok], y_fi[ok])
            pairs = sorted(zip(ctx["cols"], rf.feature_importances_), key=lambda kv: -kv[1])[:10]
            total = sum(v for _, v in pairs) or 1.0
            feature_importance_by_var[city_key] = [
                {"name": k, "value": round(float(v / total), 4)} for k, v in pairs
            ]
        except Exception as e:
            log_lines.append(f"  [FI/{city_name}/{city_key}] fail: {type(e).__name__}: {e}")

        maes = {m["name"]: m["metrics"]["mae"] for m in var_block["models"]}
        mae_str = "  ".join(f"{k}={v:.2f}" for k, v in maes.items())
        act = np.array(var_block["actual"])
        disp = {}
        for m in var_block["models"]:
            fc = np.array([v if v is not None else np.nan for v in m["forecast"]], dtype=float)
            msk = ~np.isnan(fc)
            disp[m["name"]] = float(np.mean(np.abs(fc[msk] - act[msk]))) if msk.any() else float("nan")
        disp_str = "  ".join(f"{k}={v:.2f}" for k, v in disp.items())
        lead_str = "  ".join(f"{m['name']}:h7={var_block['leadMae'][m['name']][-1]:.2f}" for m in var_block["models"])
        log_lines.append(f"   {city_key:5s} ok  |  1-step MAE: {mae_str}")
        log_lines.append(f"                |  MAE на графике (h=1..{H_MAX}, один origin — шумно): {disp_str}")
        log_lines.append(f"                |  leadMAE: {lead_str}")

    city_data = {
        "name": city_name,
        "nameEn": city_name,
        "dates": [d.strftime("%Y-%m-%d") for d in window_dates],
        "vars": vars_out,
    }
    return city_name, city_data, feature_importance_by_var, sigma_by_var, log_lines


city_var_sigma: dict[tuple, float] = {}
city_models_data: dict[str, dict] = {}
city_feature_importance: dict[str, dict] = {}
t_start = time.time()

if SPARK_DISTRIBUTED:
    try:
        _n_partitions = max(1, min(len(CITY_NAMES), SPARK_WORKER_CORES))
        _city_rows = (spark.sparkContext.parallelize(CITY_NAMES, _n_partitions)
                      .map(_run_city_benchmark).collect())
    except Exception as _e:
        try:
            spark.sparkContext.cancelAllJobs()
        except Exception:
            pass
        print(f"⚠ Spark city run failed; falling back to local execution: {type(_e).__name__}: {_e}")
        SPARK_DISTRIBUTED = False
        SPARK_DISABLE_REASON = f"city Spark task failed: {_e}"
        _city_rows = [_run_city_benchmark(c) for c in CITY_NAMES]
else:
    _city_rows = [_run_city_benchmark(c) for c in CITY_NAMES]

for _city_name, _city_data, _fi, _sigma, _logs in sorted(_city_rows, key=lambda x: x[0]):
    city_models_data[_city_name] = _city_data
    city_feature_importance[_city_name] = _fi
    city_var_sigma.update(_sigma)
    for _line in _logs:
        print(_line)
    del _city_data, _fi, _sigma, _logs
    gc.collect()

print(f"\nВремя расчёта: {(time.time() - t_start) / 60:.1f} мин")


def _ru_or_en(city_name: str) -> str:
    info = CITY_INFO.get(city_name) or {}
    return info.get("city_ru") or city_name


def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, float) and not np.isfinite(o):
        return None
    return o


cities_artifact = {
    "source":       "open-meteo archive · real data",
    "generatedAt":  datetime.now().isoformat(),
    "dates":        [d.strftime("%Y-%m-%d") for d in window_dates],
    "nPast":        CITY_PAST,
    "backStart":    CITY_BACK_START,
    "origin":       CITY_ORIGIN,
    "forecastDays": CITY_N - CITY_PAST,
    "forecastOriginDate": window_dates[CITY_ORIGIN].strftime("%Y-%m-%d"),
    "forecastFitProtocol": "monthly snapshot: refit on all observations through forecastOriginDate; no future actuals in features or fit",
    "horizons":     CITY_HORIZONS,
    "variables": [
        {"key": k, "label": v["label"], "unit": v["unit"], "icon": v["icon"]}
        for k, v in [
            ("temp", {"label": "Температура", "unit": "°C",  "icon": "🌡️"}),
            ("hum",  {"label": "Влажность",   "unit": "%",   "icon": "💧"}),
            ("rain", {"label": "Осадки",      "unit": "мм",  "icon": "🌧️"}),
            ("wind", {"label": "Ветер",       "unit": "м/с", "icon": "💨"}),
            ("pres", {"label": "Давление",    "unit": "гПа", "icon": "🧭"}),
        ]
    ],
    "modelNames": list(CITY_MODELS),
    "cities": [
        {
            "name":   _ru_or_en(cn),
            "nameEn": cn,
            "dates":  city_models_data[cn]["dates"],
            "vars":   city_models_data[cn]["vars"],
        }
        for cn in CITY_NAMES
    ],
    "featureImportance": city_feature_importance,
    "metricsSource": {
        "period":   f"{TEST_START.strftime('%Y-%m-%d')} → {TEST_END.strftime('%Y-%m-%d')}",
        "n_days":   int(((cities_daily["date"] >= TEST_START) &
                         (cities_daily["date"] <= TEST_END)).sum() // len(CITY_NAMES)),
        "protocol": (
            "Исторический benchmark: обучение до TRAIN_END, holdout TEST_START–TEST_END. "
            "Display-прогноз для выбранного origin отдельно переобучается на всей истории "
            "до origin включительно; даты после origin не используются при fit, климатологии "
            "или построении признаков. Прогноз = климатология целевой даты + прогноз аномалии. "
            "City benchmark: Persistence, Seasonal Naive, Ridge, RandomForest, "
            "XGBoost, SARIMAX, MLP, LSTM. "
            "skill = 1 − MAE_модели / MAE_Persistence на том же горизонте."
        ),
    },
}


def _clean(o):
    import math
    if isinstance(o, float):
        return o if math.isfinite(o) else None
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, list):
        return [_clean(v) for v in o]
    return o


cities_artifact = _clean(cities_artifact)

out_path = ARTIFACTS_DIR / "cities.json"
out_path.write_text(json.dumps(cities_artifact, indent=2, ensure_ascii=False))
print(f"\nsaved → {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)")
print("\nГотово.")

In [ ]:
# ============================================================
#  Визуализации city benchmark
# ============================================================
FIG_DIR = ARTIFACTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

VAR_LABELS = {"temp": "Температура, °C", "hum": "Влажность, %",
              "rain": "Осадки, мм", "wind": "Ветер, м/с",
              "pres": "Давление, гПа"}

CITY_COLORS = {
    "Almaty":   "#6366f1",
    "Moscow":   "#0ea5e9",
    "Sarajevo": "#10b981",
    "Tbilisi":  "#f59e0b",
}

if "PALETTE" not in dir():
    PALETTE = {
        "Persistence":      "#94a3b8",
        "Seasonal Naive":   "#64748b",
        "Ridge":            "#818cf8",
        "RandomForest":     "#34d399",
        "XGBoost":          "#22d3ee",
        "SARIMAX":          "#a78bfa",
        "MLP":              "#fbbf24",
        "LSTM":             "#f472b6",
    }


fig, axes = plt.subplots(1, len(CITY_VAR_MAP),
                         figsize=(4.3 * len(CITY_VAR_MAP), 4.2),
                         sharey=False)
if len(CITY_VAR_MAP) == 1:
    axes = [axes]

for ax, city_key in zip(axes, CITY_VAR_MAP):
    for city_name in CITY_NAMES:
        vb = city_models_data[city_name]["vars"][city_key]
        lead = vb["leadMae"].get("Persistence", [])[:len(CITY_HORIZONS)]
        if not lead:
            continue
        ax.plot(CITY_HORIZONS, lead,
                marker="o", ms=4, lw=2,
                color=CITY_COLORS.get(city_name, "gray"),
                label=city_name)
    ax.set_title(VAR_LABELS[city_key], fontsize=10)
    ax.set_xlabel("горизонт h, дней")
    ax.set_xticks(CITY_HORIZONS)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Lead-MAE Persistence")
axes[-1].legend(fontsize=8, loc="upper left")
fig.suptitle("Lead-MAE Persistence по городам и переменным", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "city_lead_mae_by_city.png", dpi=110, bbox_inches="tight")
plt.show()


focus_city = "Moscow" if "Moscow" in CITY_NAMES else CITY_NAMES[0]

fig, axes = plt.subplots(1, len(CITY_VAR_MAP),
                         figsize=(4.3 * len(CITY_VAR_MAP), 4.2),
                         sharey=False)
if len(CITY_VAR_MAP) == 1:
    axes = [axes]

for ax, city_key in zip(axes, CITY_VAR_MAP):
    vb = city_models_data[focus_city]["vars"][city_key]
    for m in CITY_MODELS:
        lead = vb["leadMae"].get(m, [])[:len(CITY_HORIZONS)]
        if not lead:
            continue
        ax.plot(CITY_HORIZONS, lead,
                marker="o", ms=4, lw=1.8,
                color=PALETTE.get(m, "gray"),
                linestyle="--" if m == "Persistence" else "-",
                label=m)
    ax.set_title(VAR_LABELS[city_key], fontsize=10)
    ax.set_xlabel("горизонт h, дней")
    ax.set_xticks(CITY_HORIZONS)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Lead-MAE")
axes[-1].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Lead-MAE по моделям · {CITY_INFO[focus_city]['city_ru']}",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / f"city_lead_mae_by_model_{focus_city}.png",
            dpi=110, bbox_inches="tight")
plt.show()


_lb_city = globals().get("leaderboards", {}).get("city", [])
_lb_city = [r["name"] for r in _lb_city if r["name"] != "Persistence"]
best_city_model = (_lb_city[0] if _lb_city and _lb_city[0] in CITY_MODELS
                   else ("XGBoost" if "XGBoost" in CITY_MODELS else CITY_MODELS[-1]))

fig, axes = plt.subplots(1, len(CITY_VAR_MAP),
                         figsize=(4.3 * len(CITY_VAR_MAP), 4.2),
                         sharey=False)
if len(CITY_VAR_MAP) == 1:
    axes = [axes]

for ax, city_key in zip(axes, CITY_VAR_MAP):
    for city_name in CITY_NAMES:
        vb = city_models_data[city_name]["vars"][city_key]
        lead = vb["leadMae"].get(best_city_model, [])[:len(CITY_HORIZONS)]
        if not lead:
            lead = vb["leadMae"].get("Persistence", [])[:len(CITY_HORIZONS)]
        ax.plot(CITY_HORIZONS, lead,
                marker="o", ms=4, lw=2,
                color=CITY_COLORS.get(city_name, "gray"),
                label=city_name)
    ax.set_title(VAR_LABELS[city_key], fontsize=10)
    ax.set_xlabel("горизонт h, дней")
    ax.set_xticks(CITY_HORIZONS)
    ax.grid(alpha=0.3)

axes[0].set_ylabel(f"Lead-MAE · {best_city_model}")
axes[-1].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Lead-MAE · модель {best_city_model} · сравнение городов",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "city_lead_mae_cities_best_model.png",
            dpi=110, bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(1, len(CITY_VAR_MAP),
                         figsize=(4.3 * len(CITY_VAR_MAP), 4.2),
                         sharey=True)
if len(CITY_VAR_MAP) == 1:
    axes = [axes]

for ax, city_key in zip(axes, CITY_VAR_MAP):
    for city_name in CITY_NAMES:
        vb = city_models_data[city_name]["vars"][city_key]
        base_lead = np.array(vb["leadMae"].get("Persistence", [])[:len(CITY_HORIZONS)])
        mod_lead  = np.array(vb["leadMae"].get(best_city_model, [])[:len(CITY_HORIZONS)])
        if base_lead.size == 0 or mod_lead.size == 0:
            continue
        skill = 1 - mod_lead / np.where(base_lead > 0, base_lead, 1)
        ax.plot(CITY_HORIZONS, skill * 100,
                marker="o", ms=4, lw=2,
                color=CITY_COLORS.get(city_name, "gray"),
                label=city_name)
    ax.axhline(0, color="#f43f5e", lw=1, linestyle=":")
    ax.set_title(VAR_LABELS[city_key], fontsize=10)
    ax.set_xlabel("горизонт h, дней")
    ax.set_xticks(CITY_HORIZONS)
    ax.grid(alpha=0.3)

axes[0].set_ylabel(f"Skill vs Persistence, % ({best_city_model})")
axes[-1].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Skill по горизонтам · {best_city_model} vs Persistence",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "city_skill_by_horizon.png",
            dpi=110, bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for ax, city_name in zip(axes.flat, CITY_NAMES):
    var = city_models_data[city_name]["vars"]["temp"]
    dates = city_models_data[city_name]["dates"]
    x_axis = pd.to_datetime(dates)
    actual = np.array(var["actual"])
    ax.plot(x_axis, actual, color="#0f172a", lw=2, label="Факт")
    for m_idx, m in enumerate(CITY_MODELS):
        fc = var["models"][m_idx]["forecast"]
        fc_arr = np.array([v if v is not None else np.nan for v in fc])
        ax.plot(x_axis[CITY_BACK_START:], fc_arr[CITY_BACK_START:],
                lw=1.4, alpha=0.85,
                color=PALETTE.get(m, "gray"),
                linestyle="--" if m == "Persistence" else "-",
                label=m)
    ax.set_title(CITY_INFO[city_name]["city_ru"], fontsize=10)
    ax.grid(alpha=0.3)
    ax.tick_params(axis="x", labelsize=8)
axes[0, 0].legend(fontsize=8, loc="upper left")
fig.suptitle("Прогноз vs Факт · temp · 21-дневный прогноз от origin",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "city_temp_forecast_vs_actual.png",
            dpi=110, bbox_inches="tight")
plt.show()


print(f"\nФигуры сохранены в {FIG_DIR}")

In [ ]:
# ============================================================
#  Верификация cities.json
# ============================================================
import json
from pathlib import Path

CITIES_JSON = ARTIFACTS_DIR / "cities.json"

print("=" * 64)
print("CITIES.JSON — ВЕРИФИКАЦИЯ")
print("=" * 64)

if not CITIES_JSON.exists():
    raise FileNotFoundError(
        f"{CITIES_JSON} не найден. Сначала выполни ячейку 10 (City benchmark)."
    )

data = json.loads(CITIES_JSON.read_text())
size_kb = CITIES_JSON.stat().st_size / 1024
print(f"file: {CITIES_JSON}  ({size_kb:.1f} KB)")

required_top = [
    "source", "generatedAt", "dates",
    "nPast", "backStart", "origin", "forecastDays",
    "horizons", "variables", "modelNames",
    "cities", "featureImportance",
]
missing = [k for k in required_top if k not in data]
if missing:
    print(f"  ✗ отсутствуют top-level поля: {missing}")
else:
    print(f"  ✓ все top-level поля на месте ({len(required_top)} шт.)")

print(f"\n  городов         : {len(data['cities'])}")
print(f"  моделей         : {data['modelNames']}")
expected_city_models = list(STATION_MODELS)
if data.get("modelNames") != expected_city_models:
    print(f"  ✗ список моделей не совпадает с активными backend: {expected_city_models}")
else:
    print(f"  ✓ city benchmark содержит общий набор моделей ({len(expected_city_models)}): {expected_city_models}")
if data.get("forecastDays") != 21 or data.get("nPast") != 45:
    print("  ✗ ожидается nPast=45 и forecastDays=21")
else:
    print("  ✓ окно настроено на 21 день прогноза")
print(f"  горизонтов      : {data.get('horizons')}")
print(f"  окно показа     : {data['dates'][0]} → {data['dates'][-1]}  "
      f"({len(data['dates'])} дней)")
print(f"  nPast/backStart/origin/forecastDays: "
      f"{data['nPast']}/{data['backStart']}/{data['origin']}/{data['forecastDays']}")

print("\n  По городам:")
ok_all = True
for c in data["cities"]:
    name_ru = c.get("name")
    name_en = c.get("nameEn")
    if not name_en:
        name_en = name_ru
    n_dates = len(c.get("dates", []))
    n_vars = len(c.get("vars", {}))
    vars_ok = all(
        isinstance(v, dict) and
        "actual" in v and "models" in v and
        "leadMae" in v and "metricsByHorizon" in v
        for v in c.get("vars", {}).values()
    )
    forecast_ok = all(
        sum(value is not None for value in model.get("forecast", [])) == 21
        for variable in c.get("vars", {}).values()
        for model in variable.get("models", [])
    )
    status = "✓" if (n_dates == len(data["dates"]) and vars_ok and forecast_ok) else "✗"
    if status == "✗":
        ok_all = False
    print(f"    {status} {name_ru:<15} ({name_en:<10})  "
          f"dates={n_dates}  vars={n_vars}  struct_ok={vars_ok}  21d_forecast={forecast_ok}")

print("\n  Sanity · Almaty / temp (1-step metrics, полный тест):")
c0 = next((c for c in data["cities"]
           if c.get("nameEn") == "Almaty" or c.get("name") == "Almaty"), None)
if c0 is None:
    c0 = data["cities"][0]

var = c0["vars"]["temp"]
print(f"    leadMae:")
for mname, arr in var["leadMae"].items():
    s = "  ".join(f"h{i+1}={v:.2f}" for i, v in enumerate(arr))
    print(f"      {mname:<12} {s}")

print(f"\n    metrics (1-step, полный тест):")
for m in var["models"]:
    mm = m["metrics"]
    win = m.get("metricsWindow", {})
    print(f"      {m['name']:<12} MAE={mm['mae']:.3f}  "
          f"RMSE={mm['rmse']:.3f}  R²={mm['r2']:.3f}  "
          f"skill={mm['skill']:+.3f}  (n={win.get('n', '?')})")

print(f"\n    metricsByHorizon · skill по h:")
for mname, by_h in var["metricsByHorizon"].items():
    cells = []
    for h in sorted(by_h.keys(), key=int):
        s = by_h[h].get("skill", 0.0)
        cells.append(f"h{h}={s:+.3f}")
    print(f"      {mname:<12} {'  '.join(cells)}")

print(f"\n  Sanity · sd(h) для Almaty/temp (первые 3 дня после backStart):")
for m in var["models"]:
    sd = m.get("sd") or []
    if len(sd) > data["backStart"]:
        head = sd[data["backStart"]:data["backStart"] + 3]
        print(f"    {m['name']:<12} sd[{data['backStart']}..] = {head}")

fi = data.get("featureImportance") or {}
if fi:
    top_city = next(iter(fi))
    print(f"\n  featureImportance: {len(fi)} городов × "
          f"{len(fi[top_city])} переменных (e.g. {top_city}/temp top-3):")
    for item in fi[top_city].get("temp", [])[:3]:
        print(f"    {item['name']:<15} {item['value']:.4f}")
else:
    print("\n  ✗ featureImportance пуст или отсутствует")

print("\n" + "=" * 64)
if missing:
    print("СТАТУС: ✗ cities.json неполный — сначала выполни ячейку 10")
else:
    print("СТАТУС: ✓ cities.json валиден, фронт может читать")
print("=" * 64)

In [ ]:
# ============================================================
#  Leaderboard: ожидается компонентом ModelsPage
# ============================================================
AGG_HORIZONS = [h for h in STATION_HORIZONS if h <= 10]


def _mean(xs):
    xs = [float(x) for x in xs]
    return float(np.mean(xs)) if xs else 0.0


# --- Station ranking --------------------------------------------------
station_ranking = []
for m in STATION_MODELS:
    cells_all, cells_agg = [], []
    skill_by_horizon = {h: [] for h in STATION_HORIZONS}
    for city_name in CITY_NAMES:
        evals = station_results[city_name].get(m)
        if not evals:
            continue
        for h in STATION_HORIZONS:
            cell = evals.get(h) or evals.get(str(h))
            if cell and cell.get("points") and cell.get("metrics"):
                mtr = cell["metrics"]
                cells_all.append(mtr)
                if h in AGG_HORIZONS:
                    cells_agg.append(mtr)
                skill_by_horizon[h].append(mtr["skill"])
    if not cells_all:
        continue
    station_ranking.append({
        "name": m,
        "mae":   _mean(c["mae"]  for c in cells_agg),
        "rmse":  _mean(c["rmse"] for c in cells_agg),
        "r2":    _mean(c["r2"]   for c in cells_agg),
        "skill": _mean(c["skill"] for c in cells_agg),
        "maeAllHorizons":   _mean(c["mae"]   for c in cells_all),
        "skillAllHorizons": _mean(c["skill"] for c in cells_all),
        "skillByHorizon": {
            str(h): (_mean(v) if v else 0.0) for h, v in skill_by_horizon.items()
        },
        "baseline": m in STATION_BASELINES,
    })


# --- City ranking -----------------------------------------------------
city_ranking = []
for m in CITY_MODELS:
    per_cv = []
    by_var_raw = {k: [] for k in CITY_VAR_MAP}
    by_var_skill = {k: [] for k in CITY_VAR_MAP}
    for city_name in CITY_NAMES:
        for key in CITY_VAR_MAP:
            vb = city_models_data[city_name]["vars"][key]
            by_h = vb["metricsByHorizon"].get(m)
            if not by_h:
                continue
            cells = [by_h[str(h)] for h in CITY_HORIZONS if str(h) in by_h]
            sigma = city_var_sigma.get((city_name, key), 1.0)
            per_cv.append({
                "mae":   _mean(c["mae"]  for c in cells) / sigma,
                "rmse":  _mean(c["rmse"] for c in cells) / sigma,
                "r2":    _mean(c["r2"]   for c in cells),
                "skill": _mean(c["skill"] for c in cells),
                "skill1": float(by_h["1"]["skill"]) if "1" in by_h else 0.0,
            })
            by_var_raw[key].append(_mean(c["mae"] for c in cells))
            by_var_skill[key].append(_mean(c["skill"] for c in cells))
    if not per_cv:
        continue
    city_ranking.append({
        "name":  m,
        "mae":   _mean(x["mae"]  for x in per_cv),
        "rmse":  _mean(x["rmse"] for x in per_cv),
        "r2":    _mean(x["r2"]   for x in per_cv),
        "skill": _mean(x["skill"] for x in per_cv),
        "skill1": _mean(x["skill1"] for x in per_cv),
        "byVariable": {
            k: {"mae": _mean(by_var_raw[k]), "skill": _mean(by_var_skill[k]),
                "unit": CITY_VAR_UNITS[k]}
            for k in CITY_VAR_MAP if by_var_raw[k]
        },
        "baseline": m in CITY_BASELINES,
    })

leaderboards = {
    "generatedAt": datetime.now().isoformat(),
    "aggregationHorizons": AGG_HORIZONS,
    "cityAggregationHorizons": CITY_HORIZONS,
    "note": (
        "station: mae/rmse/r2/skill усреднены по h=1..10; *AllHorizons — по всем 14. "
        "city: mae/rmse — в σ-единицах, mae/rmse/r2/skill усреднены по h=1..7."
    ),
    "station": sorted(station_ranking, key=lambda r: -r["skill"]),
    "city":    sorted(city_ranking,    key=lambda r: -r["skill"]),
}

out_path = ARTIFACTS_DIR / "leaderboards.json"
out_path.write_text(json.dumps(leaderboards, indent=2, ensure_ascii=False))
print(f"saved → {out_path}  ({out_path.stat().st_size/1024:.1f} KB)\n")

print(f"Station ranking (skill по h∈{AGG_HORIZONS} ↓):")
for r in leaderboards["station"]:
    tag = "baseline" if r["baseline"] else "        "
    print(f"  {r['name']:18s} {tag}  MAE={r['mae']:.3f}  "
          f"skill={r['skill']:+.3f}  (all-h: {r['skillAllHorizons']:+.3f})")

print(f"\nCity ranking (skill по h∈{CITY_HORIZONS} ↓; MAE в σ-единицах):")
for r in leaderboards["city"]:
    tag = "baseline" if r["baseline"] else "        "
    print(f"  {r['name']:18s} {tag}  nMAE={r['mae']:.3f}  "
          f"skill={r['skill']:+.3f}  (h=1: {r['skill1']:+.3f})")

In [ ]:
# ============================================================
#  Верификация артефактов (stations + cities + leaderboards)
# ============================================================
import json
import math

print("=" * 64)
print("ВЕРИФИКАЦИЯ АРТЕФАКТОВ")
print("=" * 64)

ok = True


def _check(cond: bool, msg_ok: str, msg_fail: str) -> None:
    global ok
    if cond:
        print(f"      ✓ {msg_ok}")
    else:
        print(f"      ✗ {msg_fail}")
        ok = False


def _fmt_size(p) -> str:
    size = p.stat().st_size / 1024
    return f"{size:.1f} KB" if size < 1024 else f"{size/1024:.1f} MB"


for fname in ["stations.json", "cities.json", "leaderboards.json"]:
    p = ARTIFACTS_DIR / fname
    if not p.exists():
        print(f"\n  ✗ {fname} — отсутствует")
        ok = False
        continue
    try:
        data = json.loads(p.read_text())
    except Exception as e:
        print(f"\n  ✗ {fname} — не парсится: {e}")
        ok = False
        continue

    print(f"\n  ✓ {fname}  ({_fmt_size(p)})")

    if fname == "stations.json":
        sd       = data["stationData"]
        models   = [m["name"] for m in data["models"]]
        horizons = data["horizons"]
        print(f"      станций    : {len(sd)}")
        print(f"      моделей    : {models}")
        expected_station_models = list(STATION_MODELS)
        _check(
            models == expected_station_models,
            f"station benchmark содержит общий набор моделей ({len(expected_station_models)})",
            f"ожидались модели: {expected_station_models}",
        )
        _check(
            len(models) == 8,
            "в stations.json сохранены 8 моделей",
            f"в stations.json сохранено {len(models)} моделей вместо 8",
        )
        print(f"      горизонтов : {len(horizons)}")

        for st in sd:
            st_label = st["station"]["label"]
            evals    = st["evaluations"]
            n_full   = 0
            for m in models:
                if m not in evals:
                    continue
                has_all = all(
                    (evals[m].get(str(h)) or evals[m].get(h))
                    and (evals[m].get(str(h)) or evals[m].get(h)).get("points")
                    for h in horizons
                )
                if has_all:
                    n_full += 1
            print(f"      · {st_label:15s} моделей с полным бэктестом: "
                  f"{n_full}/{len(models)}")
            if n_full == 0:
                ok = False

        first_st  = sd[0]["evaluations"]
        first_m   = next(iter(first_st))
        sample_h1 = first_st[first_m]["1"]
        missing = []
        if "deltaVsPersistence" not in sample_h1:
            missing.append("deltaVsPersistence")
        if "metrics" not in sample_h1 or not sample_h1["metrics"]:
            missing.append("metrics")
        if "points" not in sample_h1 or not sample_h1["points"]:
            missing.append("points")
        _check(
            not missing,
            "обязательные поля на месте (deltaVsPersistence, metrics, points)",
            f"отсутствуют поля: {missing}",
        )

        if "pointsPerHorizon" in data:
            print(f"      ✓ pointsPerHorizon: {data['pointsPerHorizon']}")

    if fname == "cities.json":
        print(f"      городов    : {len(data['cities'])}")
        print(f"      окно показа: {data['dates'][0]} → {data['dates'][-1]}  "
              f"({len(data['dates'])} дней)")
        print(f"      моделей    : {data['modelNames']}")
        expected_city_models = list(STATION_MODELS)
        _check(
            data.get("modelNames") == expected_city_models,
            f"городской benchmark содержит общий набор моделей ({len(expected_city_models)})",
            f"ожидались модели: {expected_city_models}",
        )
        _check(
            len(data.get("modelNames", [])) == 8,
            "в cities.json сохранены 8 моделей",
            f"в cities.json сохранено {len(data.get('modelNames', []))} моделей вместо 8",
        )
        _check(
            data.get("nPast") == 45 and data.get("backStart") == 45
            and data.get("origin") == 44 and data.get("forecastDays") == 21,
            "окно соответствует 45 дням до origin + 21 день прогноза",
            "ожидались nPast=45, backStart=45, origin=44, forecastDays=21",
        )
        ms = data.get("metricsSource", {})
        if ms:
            print(f"      окно метрик: {ms.get('period')}  "
                  f"({ms.get('n_days')} дней)")

        for k in ("horizons", "featureImportance", "dates"):
            _check(k in data, f"top-level '{k}' присутствует",
                   f"top-level '{k}' ОТСУТСТВУЕТ — cities.json устарел")

        if "horizons" in data:
            print(f"      горизонты  : {data['horizons']}")

        fi = data.get("featureImportance") or {}
        if fi:
            c_first = next(iter(fi))
            print(f"      featureImportance: {len(fi)} городов × "
                  f"{len(fi[c_first])} переменных")

        required_var_fields = ("actual", "models", "leadMae", "metricsByHorizon")
        for c in data["cities"]:
            name_ru = c.get("name")
            name_en = c.get("nameEn")
            _check(
                name_en is not None,
                f"nameEn '{name_en}' у {name_ru}",
                f"nameEn отсутствует у '{name_ru}'",
            )
            if name_en and name_ru and name_ru == name_en:
                print(f"      ⚠ name == nameEn = '{name_ru}' — "
                      f"русское имя не подставлено")

            for var in c["vars"]:
                vb = c["vars"][var]
                miss_var = [k for k in required_var_fields if k not in vb]
                if miss_var:
                    print(f"      ✗ {name_ru} / {var}: нет полей {miss_var}")
                    ok = False
                    continue

                n_models = len(vb["models"])
                fc0   = vb["models"][0]["forecast"]
                n_pts = sum(1 for v in fc0 if v is not None)

                has_lead = all(
                    isinstance(vb["leadMae"].get(m["name"]), list)
                    and len(vb["leadMae"][m["name"]]) == len(data.get("horizons", [1,2,3,4,5,6,7]))
                    for m in vb["models"]
                )
                has_mbh = all(
                    isinstance(vb["metricsByHorizon"].get(m["name"]), dict)
                    and len(vb["metricsByHorizon"][m["name"]]) > 0
                    for m in vb["models"]
                )
                has_metrics = all("metrics" in m and m["metrics"]
                                  for m in vb["models"])
                has_21_forecast = all(
                    sum(x is not None for x in model.get("forecast", [])) == 21
                    for model in vb["models"]
                )
                mark = "✓" if (has_metrics and has_lead and has_mbh and has_21_forecast) else "✗"
                if mark == "✗":
                    ok = False
                print(f"      {mark} {name_ru:12s} {var:5s} "
                      f"models={n_models}  fc_pts={n_pts}  "
                      f"leadMae={has_lead}  metricsByHorizon={has_mbh}  forecast21={has_21_forecast}")

                for m in vb["models"]:
                    mae = m["metrics"].get("mae", 0.0)
                    if var == "temp" and mae > 10:
                        print(f"        ⚠ {m['name']} MAE={mae:.2f} — "
                              f"подозрительно много")

                if var == "temp":
                    for mname, by_h in vb["metricsByHorizon"].items():
                        if mname == "Persistence":
                            continue
                        skills = [v.get("skill", 0.0) for v in by_h.values()]
                        if skills and all(abs(s) < 1e-9 for s in skills):
                            print(f"        ⚠ {mname}: skill=0.0 по всем h — "
                                  f"baseline не прокинут в compute_metrics")

                if var == "temp":
                    sd_arr = vb["models"][-1].get("sd") or []
                    n_sd = sum(1 for v in sd_arr if v is not None)
                    if n_sd == 0:
                        print(f"        ⚠ sd(h) пуст — resid_std не пробрасывается")

    if fname == "leaderboards.json":
        agg = data.get("aggregationHorizons", "—")
        print(f"      агрегация по h∈{agg}")
        for section in ("station", "city"):
            rows = data.get(section, [])
            print(f"      · {section}: {len(rows)} моделей")
            if not rows:
                ok = False
            else:
                top = rows[0]
                print(f"        лидер: {top['name']}  "
                      f"skill={top['skill']:+.3f}  MAE={top['mae']:.3f}")

print("\n" + "=" * 64)
if ok:
    print("ВСЁ ОК — фронт может читать:")
else:
    print("⚠ ЕСТЬ ЗАМЕЧАНИЯ — см. выше:")
print("  /artifacts/weather/cities/stations.json")
print("  /artifacts/weather/cities/cities.json")
print("  /artifacts/weather/cities/leaderboards.json")
print("=" * 64)